# Noisy Indices Detection

This notebook can be used to identify the noisy indices for all experiments, except for experiement 3 (data duplication, for which the noisy indices are the indices of the points at the end).

In [1]:
import pandas as pd
import numpy as np

In [3]:
# Run ONLY if experiment 1 (missing value error injection)

file_path = 'X_train_dirty.csv' # replace with dirty CSV file name
df = pd.read_csv(file_path)

# fill all missing values with a specific constant (-1)
df_filled_constant = df.fillna(-1)

df_filled_constant.to_csv('X_train_dirty.csv', index=False)

df_filled_constant.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol_bin
0,15.6,0.645,0.49,4.2,0.095,10.0,23.0,1.00315,2.92,0.74,1.0
1,7.4,0.620,0.05,1.9,0.068,24.0,42.0,0.99610,3.42,0.57,1.0
2,8.8,0.330,0.41,5.9,0.073,7.0,13.0,0.99658,3.30,0.62,-1.0
3,8.2,0.600,0.17,2.3,0.072,11.0,73.0,0.99630,3.20,0.45,0.0
4,10.4,0.575,0.61,2.6,0.076,11.0,24.0,1.00000,3.16,0.69,0.0


In [4]:
file1 = pd.read_csv("X_train_clean.csv") # replace with clean file (X or y train)
file2 = pd.read_csv("X_train_dirty.csv") # replace with dirty file (X or y_train)

In [5]:
def find_differing_row_indexes(df1, df2):
    """
    Reads two CSV files and returns the indexes of the rows that are different.
    A row is different if any cell value differs, or if the row exists in one file but not the other (due to differing file lengths).
    """
    # align columns for comparison based on common columns
    common_cols = list(set(df1.columns) & set(df2.columns))
    if not common_cols:
        print("Error: Files have no common columns to compare.")
        return []

    df1_comp = df1[common_cols]
    df2_comp = df2[common_cols]

    # find minimum number of rows to compare content
    min_rows = min(len(df1_comp), len(df2_comp))
    df1_comp_aligned = df1_comp.iloc[:min_rows]
    df2_comp_aligned = df2_comp.iloc[:min_rows]

    # 1. find differences in the overlapping rows (aligned by index)
    differing_indexes_bool = (df1_comp_aligned != df2_comp_aligned).any(axis=1)

    # the indexes of the differing rows (up to min_rows)
    diff_indices_aligned = df1_comp_aligned.index[differing_indexes_bool].tolist()

    # 2. handle rows that exist only in one file (if lengths are different)
    if len(df1) > len(df2):
        extra_df1_indices = df1.index[min_rows:].tolist()
        diff_indices_aligned.extend(extra_df1_indices)
    elif len(df2) > len(df1):
        extra_df2_indices = df2.index[min_rows:].tolist()
        diff_indices_aligned.extend(extra_df2_indices)

    # return sorted, unique indexes
    return sorted(list(set(diff_indices_aligned)))

In [6]:
differing_indexes = find_differing_row_indexes(file1, file2)
print(f"Indices of differing rows: {differing_indexes}")

Indices of differing rows: [2, 7, 9, 21, 39, 58, 63, 94, 116, 139, 148, 150, 151, 163, 189, 196, 208, 213, 214, 221, 223, 230, 231, 238, 254, 276, 281, 286, 292, 299, 308, 312, 325, 343, 344, 353, 354, 356, 363, 366, 367, 376, 379, 386, 388, 389, 416, 419, 424, 435, 436, 439, 449, 454, 464, 473, 481, 484, 488, 511, 514, 516, 522, 527, 541, 543, 549, 563, 569, 571, 575, 577, 578, 580, 581, 598, 607, 623, 630, 718, 724, 731, 744, 754, 768, 776, 785, 791, 804, 816, 822, 839, 843, 847, 851, 859, 866, 867, 868, 882, 891, 893, 905, 910, 920, 924, 925, 933, 939, 949, 966, 973, 986, 988, 997, 1000, 1001, 1014, 1024, 1026, 1032, 1038, 1043, 1058, 1072, 1074, 1076, 1107, 1111, 1121, 1127, 1134, 1136, 1146, 1162, 1168, 1174, 1176, 1182, 1185, 1199, 1207, 1212, 1239, 1252, 1255, 1258, 1259, 1268, 1269, 1278]


In [7]:
file_name = "noisy_indices.txt"

with open(file_name, 'w') as file:
    file.write(str(differing_indexes))

print(f"Content written to '{file_name}' successfully.")

Content written to 'noisy_indices.txt' successfully.
